# 🛠️ Notebook 1 — Feature Extractor (`core/feature_extractor.py`)

**What this file does:** It is the "eyes" of our defect detector. It takes product images, pushes them through a pretrained CNN (WideResNet50), and pulls out the internal feature maps that describe what the network "sees".

**Why it exists:** PatchCore doesn't train anything — it only *extracts* features from a frozen backbone. This file handles all the extraction, smoothing, alignment, and flattening logic.

**How it fits in the pipeline:**
```
dataset.py → THIS FILE → coreset.py → memory_bank.py
```
We load images, extract rich feature vectors for every small patch of every image, and pass them downstream for storage.

## 1 — Setup
Mount Google Drive, install dependencies, unzip the datasets, and configure logging.
After this cell you will have:
- `bottle`, `carpet`, `screw` folders under `/content/mvtec/`
- All required Python packages ready to import

In [ ]:
# ---- Mount Google Drive ----
from google.colab import drive
drive.mount('/content/drive')

# ---- Install missing packages ----
!pip install -q faiss-gpu tqdm

# ---- Unzip datasets ----
import zipfile, glob, os, logging

DRIVE_DATA = '/content/drive/MyDrive/defects-detection-CV'
MVTEC_ROOT = '/content/mvtec'
os.makedirs(MVTEC_ROOT, exist_ok=True)

for zf in glob.glob(os.path.join(DRIVE_DATA, '*.zip')):
    name = os.path.basename(zf).split('-')[0]          # 'bottle', 'carpet', or 'screw'
    dest = os.path.join(MVTEC_ROOT, name)
    if os.path.isdir(dest):
        print(f'{name} already unzipped, skipping.')
        continue
    print(f'Unzipping {name}...')
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(MVTEC_ROOT)
    print(f'  done → {dest}')

# ---- Logging (replaces print everywhere) ----
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('feature_extractor')
logger.info('Setup complete.')

## 2 — Configuration
All tuneable values live here so nothing is hard-coded inside functions.
Change a value once and every function picks it up automatically.

In [ ]:
import torch
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    """Central place for every hyper-parameter."""
    # Paths
    DATA_ROOT: str = '/content/mvtec'
    DRIVE_DATA: str = '/content/drive/MyDrive/defects-detection-CV'
    OUTPUT_DIR: str = '/content/drive/MyDrive/defects-detection-CV/outputs'

    # Data
    CATEGORIES: List[str] = field(default_factory=lambda: ['bottle', 'carpet', 'screw'])
    IMAGE_SIZE: int = 224
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 2

    # Backbone
    BACKBONE: str = 'wide_resnet50_2'
    LAYERS: List[str] = field(default_factory=lambda: ['layer2', 'layer3'])

    # Coreset
    CORESET_RATIO: float = 0.01

    # Checkpoints
    CHECKPOINT_EVERY: int = 10    # save progress every N batches

    # Device
    DEVICE: str = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
logger.info('Device: %s', cfg.DEVICE)

## 3 — Dataset & Backbone helpers
We need a DataLoader that reads training images and a backbone model
with hooks that capture layer2 and layer3 feature maps.
These would normally live in `data/dataset.py` and `models/backbone.py`,
but we define them inline so this notebook is fully self-contained.

In [ ]:
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from typing import Dict, Tuple

# --------------- Dataset ---------------
class MVTecTrainDataset(Dataset):
    """Loads only the GOOD (defect-free) training images for one category."""

    def __init__(self, root: str, category: str, image_size: int) -> None:
        self.image_dir = os.path.join(root, category, 'train', 'good')
        self.paths = sorted([
            os.path.join(self.image_dir, f)
            for f in os.listdir(self.image_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        logger.info('Dataset [%s]: %d images from %s', category, len(self.paths), self.image_dir)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> torch.Tensor:
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img)


def get_train_dataloader(category: str) -> DataLoader:
    ds = MVTecTrainDataset(cfg.DATA_ROOT, category, cfg.IMAGE_SIZE)
    return DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True)


# --------------- Backbone ---------------
def load_backbone(device: str) -> Tuple[torch.nn.Module, Dict[str, torch.Tensor]]:
    """Load a frozen WideResNet50 and register hooks on layer2 & layer3."""
    model = getattr(models, cfg.BACKBONE)(weights='IMAGENET1K_V1')
    model = model.to(device).eval()
    for p in model.parameters():
        p.requires_grad = False

    hook_dict: Dict[str, torch.Tensor] = {}

    def _hook(name: str):
        def fn(module, inp, out):
            hook_dict[name] = out.detach()
        return fn

    model.layer2.register_forward_hook(_hook('layer2'))
    model.layer3.register_forward_hook(_hook('layer3'))
    logger.info('Backbone loaded with hooks on layer2 & layer3.')
    return model, hook_dict

# Quick test
model, hook_dict = load_backbone(cfg.DEVICE)

---
# 🔍 `core/feature_extractor.py`
Below are the **5 functions** that make up the feature extraction pipeline.
Each one does one small job, and they chain together like an assembly line:

```
image → extract_layer_features → locally_aware_patches → align_and_concat → flatten_patches
```
`extract_dataset_features` orchestrates all four over every image in the dataset.

### Function 1 — `extract_layer_features`
Runs one forward pass through the backbone and grabs the feature maps
that the hooks captured from **layer2** (fine details like edges) and
**layer3** (higher-level shapes). Think of it as taking an X-ray of
the image at two different zoom levels.

In [ ]:
import torch.nn.functional as F
from tqdm import tqdm

def extract_layer_features(
    model: torch.nn.Module,
    image_batch: torch.Tensor,
    hook_dict: Dict[str, torch.Tensor],
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Run a forward pass and return the captured layer2 & layer3 maps.

    Args:
        model: Frozen WideResNet50 backbone (already on device).
        image_batch: Batch of images, shape (B, 3, 224, 224).
        hook_dict: Dict populated by hooks with feature maps.

    Returns:
        (layer2_feats, layer3_feats)
        layer2: (B, 512, 28, 28)
        layer3: (B, 1024, 14, 14)
    """
    with torch.no_grad():
        _ = model(image_batch)

    feat_l2 = hook_dict['layer2']
    feat_l3 = hook_dict['layer3']
    logger.debug('Extracted — layer2: %s, layer3: %s', feat_l2.shape, feat_l3.shape)
    return feat_l2, feat_l3

### Function 2 — `locally_aware_patches`
A single pixel in the feature map knows about only a tiny part of the image.
By averaging each pixel with its 3×3 neighbours, we give it **local context** —
it now also knows what surrounds it. This makes defect detection more robust
because defects rarely sit on a single pixel.

Implementation: a simple `avg_pool2d` with stride=1 and same-padding so the
output shape stays identical to the input.

In [ ]:
def locally_aware_patches(
    feature_map: torch.Tensor,
    patch_size: int = 3,
) -> torch.Tensor:
    """Smooth each spatial position by averaging its neighbourhood.

    Args:
        feature_map: (B, C, H, W) tensor.
        patch_size: Size of the averaging window (default 3).

    Returns:
        Locally-aware feature map, same shape (B, C, H, W).
    """
    padding = patch_size // 2
    result = F.avg_pool2d(
        feature_map,
        kernel_size=patch_size,
        stride=1,
        padding=padding,
    )
    logger.debug('Locally-aware patches applied (kernel=%d)', patch_size)
    return result

### Function 3 — `align_and_concat`
layer2 is 28×28 (fine detail), layer3 is 14×14 (coarse shapes).
We **upsample** layer3 to 28×28 so both maps have the same grid,
then **concatenate** them along the channel axis.

Result: each position now has **1536 channels** (512 + 1024) of
combined fine + coarse information.

In [ ]:
def align_and_concat(
    feat_layer2: torch.Tensor,
    feat_layer3: torch.Tensor,
) -> torch.Tensor:
    """Upsample layer3 to layer2 resolution and concatenate.

    Args:
        feat_layer2: (B, 512, 28, 28)
        feat_layer3: (B, 1024, 14, 14)

    Returns:
        Combined feature map (B, 1536, 28, 28).
    """
    h, w = feat_layer2.shape[2], feat_layer2.shape[3]
    feat_l3_up = F.interpolate(
        feat_layer3, size=(h, w), mode='bilinear', align_corners=False,
    )
    combined = torch.cat([feat_layer2, feat_l3_up], dim=1)
    logger.debug('Aligned & concatenated → %s', combined.shape)
    return combined

### Function 4 — `flatten_patches`
Reshapes the 4-D feature map into a simple 2-D **table** where each row
is one patch vector. For a batch of 32 images with a 28×28 grid that
gives us 32 × 784 = 25,088 rows, each 1536 values long.

We also return the spatial shape (H, W) so we can later reshape back
to a 2-D heatmap for visualisation.

In [ ]:
def flatten_patches(
    feature_map: torch.Tensor,
) -> Tuple[torch.Tensor, Tuple[int, int]]:
    """Reshape (B, C, H, W) → (B*H*W, C).

    Args:
        feature_map: (B, C, H, W) tensor.

    Returns:
        flat: (B*H*W, C) tensor.
        spatial_shape: (H, W) for later un-flattening.
    """
    B, C, H, W = feature_map.shape
    flat = feature_map.permute(0, 2, 3, 1).reshape(-1, C)
    logger.debug('Flattened %s → %s', feature_map.shape, flat.shape)
    return flat, (H, W)

### Function 5 — `extract_dataset_features`
The **main workhorse**. It loops over every batch from the DataLoader,
calls the four functions above in sequence, and collects all patch
vectors into one big tensor.

Includes a **checkpoint system**: every N batches the collected features
are saved to disk so we can resume if Colab disconnects. Features are
kept on CPU to avoid running out of GPU memory.

In [ ]:
def extract_dataset_features(
    model: torch.nn.Module,
    dataloader: DataLoader,
    hook_dict: Dict[str, torch.Tensor],
    device: str,
    checkpoint_path: str = '/content/drive/MyDrive/defects-detection-CV/outputs/feat_ckpt.pt',
    checkpoint_every: int = 10,
) -> torch.Tensor:
    """Extract features for every training image.

    Args:
        model: Frozen backbone on device.
        dataloader: Training dataloader (good images only).
        hook_dict: Hook dictionary attached to the model.
        device: 'cuda' or 'cpu'.
        checkpoint_path: Where to save intermediate progress.
        checkpoint_every: Save a checkpoint every N batches.

    Returns:
        Tensor of shape (N_total_patches, 1536) on CPU.
    """
    all_features: list = []
    start_batch: int = 0

    # --- Try to resume from a previous checkpoint ---
    try:
        ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        all_features = ckpt['features']
        start_batch = ckpt['next_batch']
        logger.info('Resumed from checkpoint at batch %d', start_batch)
    except FileNotFoundError:
        logger.info('No checkpoint found — starting fresh.')
    except Exception as exc:
        logger.warning('Could not load checkpoint (%s) — starting fresh.', exc)

    batches = list(dataloader)
    remaining = batches[start_batch:]

    for batch_idx, images in enumerate(
        tqdm(remaining, desc='Extracting features',
             initial=start_batch, total=len(batches)),
        start=start_batch,
    ):
        images = images.to(device)

        # 1 — forward pass
        feat_l2, feat_l3 = extract_layer_features(model, images, hook_dict)

        # 2 — locally-aware smoothing
        feat_l2 = locally_aware_patches(feat_l2)
        feat_l3 = locally_aware_patches(feat_l3)

        # 3 — align sizes & concatenate channels
        combined = align_and_concat(feat_l2, feat_l3)

        # 4 — flatten to (num_patches, C), keep on CPU
        flat, _ = flatten_patches(combined)
        all_features.append(flat.cpu())

        # 5 — checkpoint
        if (batch_idx + 1) % checkpoint_every == 0:
            try:
                torch.save(
                    {'features': all_features, 'next_batch': batch_idx + 1},
                    checkpoint_path,
                )
                logger.info('Checkpoint saved at batch %d', batch_idx + 1)
            except Exception as exc:
                logger.warning('Checkpoint save failed: %s', exc)

    result = torch.cat(all_features, dim=0)
    logger.info('Total patches extracted: %s', result.shape)

    # Clean up checkpoint file after successful completion
    try:
        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)
            logger.info('Checkpoint file cleaned up.')
    except Exception as exc:
        logger.warning('Could not remove checkpoint: %s', exc)

    return result

---
## ▶️ Run the Feature Extractor
Let’s extract features for each category and save them to Google Drive.
Each category produces a `.pt` file containing a tensor of shape
`(N_patches, 1536)` — these are the raw features that the next notebook
(coreset sampling) will compress.

In [ ]:
FEATURES_DIR = os.path.join(cfg.OUTPUT_DIR, 'raw_features')
os.makedirs(FEATURES_DIR, exist_ok=True)

for category in cfg.CATEGORIES:
    save_path = os.path.join(FEATURES_DIR, f'{category}_features.pt')

    # Skip if already extracted
    if os.path.exists(save_path):
        logger.info('[%s] Features already exist at %s — skipping.', category, save_path)
        continue

    logger.info('=== Processing category: %s ===', category)
    dataloader = get_train_dataloader(category)

    ckpt_path = os.path.join(FEATURES_DIR, f'{category}_feat_ckpt.pt')
    features = extract_dataset_features(
        model, dataloader, hook_dict, cfg.DEVICE,
        checkpoint_path=ckpt_path,
        checkpoint_every=cfg.CHECKPOINT_EVERY,
    )

    try:
        torch.save(features, save_path)
        logger.info('[%s] Saved features %s to %s', category, features.shape, save_path)
    except Exception as exc:
        logger.error('[%s] Failed to save features: %s', category, exc)

logger.info('All categories done!')

## ✅ Summary
At this point, Google Drive should contain:
```
defects-detection-CV/outputs/raw_features/
   bottle_features.pt    ← (~160k patches, 1536 dims)
   carpet_features.pt    ← (~220k patches, 1536 dims)
   screw_features.pt     ← (~250k patches, 1536 dims)
```
The next notebook (`02_coreset_and_memory_bank`) will load these files,
compress them with coreset sampling, and build the final memory bank + FAISS index.